In [ ]:
%matplotlib inline

from pathlib import Path

import jax
import jax.numpy as jnp
import jax.typing as jt
import jax.experimental.checkify as checkify
import kagglehub
import matplotlib.pyplot as plt

from reimpl_a_gn.dataset.synthetic_nerf_dataset import load_synthetic_nerf_dataset
from reimpl_a_gn.threed.plotting import plot_cameras
from reimpl_a_gn.threed.rendering import (
    CameraParams,
    compute_nerf_positional_encoding,
    sample_nerf_rendering_positions_along_rays,
    sample_regular_positions_along_rays,
)

In [ ]:
def get_flower_dataset_path():
    """Download the dataset if needed, and return the path to the flower subfolder."""
    return (
        Path(kagglehub.dataset_download("arenagrenade/llff-dataset-full"))
        / "nerf_llff_data"
        / "flower"
    )

In [ ]:
def plot_pos_encoding():
    # print(compute_nerf_positional_encoding(jnp.array([[2, 3, 4.5, 0.5, 1.0, 0.3]]), 4))

    plot_min_val, plot_max_val = -1.3, 1.3
    plot_sample_count = 1000
    component_count = 5
    sample_plot_points = jnp.array([-1.0, -0.7, -0.3, -0.05, 0.2, 0.5, 0.9])

    fig, axes = plt.subplots(5, 2)
    fig.set_size_inches(12, 12)
    fig.suptitle(
        f"NeRF positional encoding with {component_count} components\n"
        f"(encoding values: {', '.join([str(x) for x in sample_plot_points])})",
        fontsize=16,
    )

    base_plot_points = jnp.linspace(plot_min_val, plot_max_val, plot_sample_count)
    for power_of_two in range(component_count):

        def f1(z):
            return jnp.sin(jnp.pow(2, power_of_two) * jnp.pi * z)

        def f2(z):
            return jnp.cos(jnp.pow(2, power_of_two) * jnp.pi * z)

        # plot trig functions
        f1_plot_vals = f1(base_plot_points)
        f2_plot_vals = f2(base_plot_points)
        axes[power_of_two, 0].plot(base_plot_points, f1_plot_vals)
        axes[power_of_two, 1].plot(base_plot_points, f2_plot_vals)

        axes[power_of_two, 0].set_xticks(sample_plot_points)
        axes[power_of_two, 1].set_xticks(sample_plot_points)
        axes[power_of_two, 0].vlines(sample_plot_points, -1.5, 1.5, color="orange")
        axes[power_of_two, 1].vlines(sample_plot_points, -1.5, 1.5, color="orange")

        sample_f1_values = f1(sample_plot_points)
        sample_f2_values = f2(sample_plot_points)
        axes[power_of_two, 0].scatter(sample_plot_points, sample_f1_values, color="red")
        axes[power_of_two, 1].scatter(sample_plot_points, sample_f2_values, color="red")

        axes[power_of_two, 0].set_title(f"sin(2^{power_of_two} * pi * x)", loc="left")
        axes[power_of_two, 1].set_title(f"cos(2^{power_of_two} * pi * x)", loc="right")

    fig.tight_layout(pad=0.1)
    plt.show()

    y = jnp.array([[2, 3, 4.5, 0.5, 1.0, 0.3]])
    components = 3
    result = jnp.zeros(list(y.shape[:-1]) + [6, 2 * components], dtype=float)
    for power_of_two in range(components):
        result = result.at[..., power_of_two * 2].set(
            jnp.sin(jnp.pow(2, power_of_two) * jnp.pi * y)
        )
        result = result.at[..., power_of_two * 2 + 1].set(
            jnp.cos(jnp.pow(2, power_of_two) * jnp.pi * y)
        )

    y = jnp.array([[0.4, 0.6, 0.1, 0.5, 0.3, 0.8]], dtype=float)
    pe = compute_nerf_positional_encoding(y, 12)
    print(pe)

# plot_pos_encoding()

In [ ]:
CAMERA_ORIGIN_IN_WORLD_FRAME = jnp.array([3.0, 2.5, 4.0, 1.0])
CAMERA_FRAME_DIRECTIONS = jnp.array(
    [[-0.5, -0.5, -0.5], [-0.5, 0.5, -0.5], [-0.5, 0.0, 0.5]]
)
CAMERA_TO_WORLD_MATRIX = jnp.zeros((4, 4))
CAMERA_TO_WORLD_MATRIX = CAMERA_TO_WORLD_MATRIX.at[:3, :3].set(
    CAMERA_FRAME_DIRECTIONS.transpose()
)
CAMERA_TO_WORLD_MATRIX = CAMERA_TO_WORLD_MATRIX.at[:, 3].set(CAMERA_ORIGIN_IN_WORLD_FRAME)
WORLD_TO_CAMERA_MATRIX = jnp.linalg.inv(CAMERA_TO_WORLD_MATRIX)
CAMERA_INTRINSICS = jnp.array([[0.1, 0, 1], [0, 0.1, 1], [0, 0, 1]])

print("camera to world transformation:")
print(CAMERA_TO_WORLD_MATRIX)
print("world to camera transformation:")
print(WORLD_TO_CAMERA_MATRIX)
print("camera intrinsic matrix:")
print(CAMERA_INTRINSICS)

In [ ]:
def plot_coordinate_systems(camera_origin, camera_directions, ax):
    def plot_basis(origin: jt.ArrayLike, dxdydz: jt.ArrayLike, plt_axis):
        """Plot quivers representing an orthonormal basis in 3D.

        @param origin Origin of the coordinate system in world coordinates.
        @param dxdydz Mutually orthogonal vectors representing the axes. Shape: (3, 3).
        Dimensions: vector, then vector coordinates.
        """

        origin = jnp.array(origin)
        assert origin.shape == (4,)
        dxdydz = jnp.array(dxdydz)
        assert dxdydz.shape == (3, 3)
        for direction in range(3):
            plt_axis.quiver(
                origin[0] / origin[3],
                origin[1] / origin[3],
                origin[2] / origin[3],
                dxdydz[direction, 0],
                dxdydz[direction, 1],
                dxdydz[direction, 2],
                normalize=True,
            )

    plot_basis(camera_origin, camera_directions, ax)


def plot_our_camera():
    ax = plt.figure().add_subplot(projection="3d", aspect="equal")
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_zlim(-5, 5)  # type: ignore

    plot_coordinate_systems(CAMERA_ORIGIN_IN_WORLD_FRAME, CAMERA_FRAME_DIRECTIONS, ax)
    plt.show()

plot_our_camera()

In [ ]:
# plot points in world, then camera frame to check that that transform works well
def plot_points_in_world_and_camera_frames():
    print("camera origin in world frame:")
    print(CAMERA_ORIGIN_IN_WORLD_FRAME)

    points_in_world_frame = jnp.array(
        [[2, 3, 4, 1], [7.5, 6, -3, 3], [-2, -0.5, 1, 0.7]], dtype=float
    )
    print("original points in world frame:")
    print(points_in_world_frame)

    fig, ((world_ax, camera_ax), (inv_ax, jc_ax)) = plt.subplots(
        2, 2, subplot_kw={"projection": "3d", "aspect": "equal"}
    )
    fig.set_size_inches(15, 15)
    world_ax.set_title("points in world frame\n(origin in red, camera in green)")
    world_ax.scatter(0, 0, 0, color="red")
    world_ax.scatter(
        CAMERA_ORIGIN_IN_WORLD_FRAME[0] / CAMERA_ORIGIN_IN_WORLD_FRAME[3],
        CAMERA_ORIGIN_IN_WORLD_FRAME[1] / CAMERA_ORIGIN_IN_WORLD_FRAME[3],
        CAMERA_ORIGIN_IN_WORLD_FRAME[2] / CAMERA_ORIGIN_IN_WORLD_FRAME[3],
        color="green",
    )
    world_ax.scatter(
        points_in_world_frame[:, 0] / points_in_world_frame[:, 3],
        points_in_world_frame[:, 1] / points_in_world_frame[:, 3],
        points_in_world_frame[:, 2] / points_in_world_frame[:, 3],
        color="blue",
    )

    camera_ax.set_title(
        "points in camera frame\n(transformed, origin in green, world origin in red)"
    )
    world_origin_in_camera_frame = jnp.array([[0, 0, 0, 1]]) @ WORLD_TO_CAMERA_MATRIX.T
    points_in_camera_frame_transformed = (
        points_in_world_frame @ WORLD_TO_CAMERA_MATRIX.T
    )
    print("points transformed to camera frame:")
    print(points_in_camera_frame_transformed)
    camera_ax.scatter(
        world_origin_in_camera_frame[:, 0] / world_origin_in_camera_frame[:, 3],
        world_origin_in_camera_frame[:, 1] / world_origin_in_camera_frame[:, 3],
        world_origin_in_camera_frame[:, 2] / world_origin_in_camera_frame[:, 3],
        color="red",
    )
    camera_ax.scatter([0], [0], [0], color="green")
    camera_ax.scatter(
        points_in_camera_frame_transformed[:, 0]
        / points_in_camera_frame_transformed[:, 3],
        points_in_camera_frame_transformed[:, 1]
        / points_in_camera_frame_transformed[:, 3],
        points_in_camera_frame_transformed[:, 2]
        / points_in_camera_frame_transformed[:, 3],
        color="blue",
    )

    inv_ax.set_title("points in world frame\n(transformed back, origin in red, camera in green)")
    points_in_world_frame_transformed_back = (
        points_in_camera_frame_transformed @ CAMERA_TO_WORLD_MATRIX.T
    )
    inv_ax.scatter(0, 0, 0, color="red")
    inv_ax.scatter(
        CAMERA_ORIGIN_IN_WORLD_FRAME[0] / CAMERA_ORIGIN_IN_WORLD_FRAME[3],
        CAMERA_ORIGIN_IN_WORLD_FRAME[1] / CAMERA_ORIGIN_IN_WORLD_FRAME[3],
        CAMERA_ORIGIN_IN_WORLD_FRAME[2] / CAMERA_ORIGIN_IN_WORLD_FRAME[3],
        color="green",
    )
    inv_ax.scatter(
        points_in_world_frame_transformed_back[:, 0]
        / points_in_world_frame_transformed_back[:, 3],
        points_in_world_frame_transformed_back[:, 1]
        / points_in_world_frame_transformed_back[:, 3],
        points_in_world_frame_transformed_back[:, 2]
        / points_in_world_frame_transformed_back[:, 3],
    )
    jc_ax.set_title("plot by john cage")


plot_points_in_world_and_camera_frames()

In [ ]:
def plot_sampled_rays(world_to_camera, camera_intrinsics):
    camera_params = CameraParams(
        world_to_camera,
        camera_intrinsics,
    )

    print("camera intrinsics:")
    print(camera_intrinsics)

    camera_origin_camera = jnp.array([[0, 0, 0, 1]])
    camera_origin_world = (
        camera_origin_camera @ camera_params.camera_to_world.T
    ).squeeze(0)

    # pixel grid
    all_x, all_y = jnp.meshgrid(jnp.arange(-2, 3, 0.7), jnp.arange(-5, 5, 0.5))
    grid_points_image = jnp.stack([all_x, all_y], axis=-1).reshape(-1, 2)
    grid_ray_directions_camera = camera_params.image_to_camera(grid_points_image)
    print("first ray directions, camera frame:")
    print(grid_ray_directions_camera[:10])
    grid_ray_directions_world = camera_params.image_to_world(grid_points_image)
    print("first ray directions, world frame:")
    print(grid_ray_directions_world[:10])

    # rays we sample on
    rays_camera = jnp.concatenate(
        [
            # origins at zero
            jnp.full(
                [grid_ray_directions_camera.shape[0], 4],
                jnp.array([[0, 0, 0, 1]]),
            ),
            # direction vectors are just the point coordinates
            grid_ray_directions_camera,
        ],
        axis=1,
    )

    err, sampled_positions_regular_camera = checkify.checkify(
        sample_regular_positions_along_rays
    )(rays_camera, rays_camera.shape[0], 0.1, 5, 10)
    err.throw()
    sampled_positions_regular_camera = sampled_positions_regular_camera.at[:, 3].set(
        -sampled_positions_regular_camera[:, 3]
    )
    sampled_positions_regular_world = (
        sampled_positions_regular_camera @ camera_params.camera_to_world.T
    )
    fig_rays, axes = plt.subplots(
        2, 2, subplot_kw={"projection": "3d", "aspect": "equal"}
    )
    fig_rays.set_size_inches(15, 15)
    ((ax_rays_regular, ax_rays_nerf), (ax_ray_directions_camera, ax_ray_directions_world)) = axes

    ax_rays_regular.set_title("rays towards pixels, regular sampling")
    ax_rays_regular.text(1, 0, 0, "x-axis", "x")  # type: ignore
    ax_rays_regular.text(0, 1, 0, "y-axis", "y")  # type: ignore
    ax_rays_regular.text(0, 0, 1, "z-axis", "z")  # type: ignore
    assert sampled_positions_regular_world.shape[-1] == 4
    sampled_positions_regular_world_flat = sampled_positions_regular_world.reshape(
        -1, 4
    )
    # positions on rays in blue
    ax_rays_regular.scatter(
        sampled_positions_regular_world_flat[:, 0]
        / sampled_positions_regular_world_flat[:, 3],
        sampled_positions_regular_world_flat[:, 1]
        / sampled_positions_regular_world_flat[:, 3],
        sampled_positions_regular_world_flat[:, 2]
        / sampled_positions_regular_world_flat[:, 3],
        color="blue",
    )
    # image pixels in orange
    ax_rays_regular.scatter(
        grid_ray_directions_world[:, 0],
        grid_ray_directions_world[:, 1],
        grid_ray_directions_world[:, 2],
        color="orange",
    )
    # camera origin in red
    ax_rays_regular.scatter(
        camera_origin_world[0] / camera_origin_world[3],
        camera_origin_world[1] / camera_origin_world[3],
        camera_origin_world[2] / camera_origin_world[3],
        color="red",
    )

    err, sampled_positions_nerf_camera = checkify.checkify(
        sample_nerf_rendering_positions_along_rays
    )(rays_camera, rays_camera.shape[0], 0.5, 3.0, 3, jax.random.PRNGKey(0))
    err.throw()
    ax_rays_nerf.set_title("rays towards pixels, NeRF (bins) sampling")
    ax_rays_nerf.text(1, 0, 0, "x-axis", "x")  # type: ignore
    ax_rays_nerf.text(0, 1, 0, "y-axis", "y")  # type: ignore
    ax_rays_nerf.text(0, 0, 1, "z-axis", "z")  # type: ignore
    assert sampled_positions_nerf_camera.shape[-1] == 4
    sampled_positions_nerf_camera_flat = sampled_positions_nerf_camera.reshape(-1, 4)
    sampled_positions_nerf_world_flat = (
        sampled_positions_nerf_camera_flat @ camera_params.camera_to_world.T
    )
    # positions on rays in blue
    ax_rays_nerf.scatter(
        sampled_positions_nerf_world_flat[:, 0]
        / sampled_positions_nerf_world_flat[:, 3],
        sampled_positions_nerf_world_flat[:, 1]
        / sampled_positions_nerf_world_flat[:, 3],
        sampled_positions_nerf_world_flat[:, 2]
        / sampled_positions_nerf_world_flat[:, 3],
        color="blue",
    )
    # image pixels in orange
    ax_rays_nerf.scatter(
        grid_ray_directions_world[:, 0],
        grid_ray_directions_world[:, 1],
        grid_ray_directions_world[:, 2],
        color="orange",
    )
    # camera origin in red
    ax_rays_nerf.scatter(
        camera_origin_world[0] / camera_origin_world[3],
        camera_origin_world[1] / camera_origin_world[3],
        camera_origin_world[2] / camera_origin_world[3],
        color="red",
    )

    plot_coordinate_systems(
        CAMERA_ORIGIN_IN_WORLD_FRAME, CAMERA_FRAME_DIRECTIONS, ax_rays_regular
    )
    plot_coordinate_systems(CAMERA_ORIGIN_IN_WORLD_FRAME, CAMERA_FRAME_DIRECTIONS, ax_rays_nerf)

    ax_ray_directions_camera.set_title("Ray directions, camera coordinates\n(origin in red)")
    ax_ray_directions_camera.scatter(*(grid_ray_directions_camera[:3]), color="green")
    ax_ray_directions_camera.scatter(0, 0, 0, color="red")
    ax_ray_directions_world.set_title("Ray directions, world coordinates\n(origin in red)")
    ax_ray_directions_world.scatter(*(grid_ray_directions_world[:3]), color="green")
    ax_ray_directions_world.scatter(0, 0, 0, color="red")

    plt.show()


plot_sampled_rays(WORLD_TO_CAMERA_MATRIX, CAMERA_INTRINSICS)

In [ ]:
def test_load_data():
    data = load_synthetic_nerf_dataset(get_flower_dataset_path())
    for array_name in ("images", "bds", "poses", "render_poses"):
        print(f"shape of {array_name} array: {getattr(data, array_name).shape}")
    print(f"holdout image index: {data.i_test}")
    plot_cameras(data.cameras, data.i_test)

test_load_data()

In [ ]:
def test_sample_rays_from_actual_cameras():
    data = load_synthetic_nerf_dataset(get_flower_dataset_path())
    camera_params = data.cameras[0]

    all_x, all_y = jnp.meshgrid(jnp.arange(-2, 3, 0.7), jnp.arange(-5, 5, 0.5))
    all_pixel_xy = jnp.stack([all_x, all_y], axis=1).reshape((-1, 2))
    ray_origins_camera_frame = jnp.zeros([all_pixel_xy.shape[0], 4], dtype=float).at[:, 3].set(1)
    ray_directions_camera_frame = camera_params.image_to_camera(all_pixel_xy)
    rays_camera_frame = jnp.concat([ray_origins_camera_frame, ray_directions_camera_frame], axis=1)
    assert len(rays_camera_frame.shape) == 2
    assert rays_camera_frame.shape[1] == 8

    # we will work with two axes and reshape at the end
    rays_world_frame = jnp.zeros(rays_camera_frame.shape)
    print(f"rays shape: {rays_world_frame.shape}")
    # transform ray origins
    rays_world_frame = rays_world_frame.at[:, :4].set(
        rays_camera_frame[:, :4] @ camera_params.camera_to_world.T
    )
    # transform ray directions
    rays_world_frame = rays_world_frame.at[:, 4:8].set(
        rays_camera_frame[..., 4:8] @ camera_params.camera_to_world.T
    )

    def plot_pixels(ax: plt.Axes, points: jnp.ndarray):  # type: ignore
        ax.scatter(points[:, 0], points[:, 1], points[:, 2])

    err, regularly_sampled_rays = checkify.checkify(
        sample_regular_positions_along_rays
    )(
        rays_world_frame,
        rays_camera_frame.shape[0],
        0.5,
        5,
        7,
    )
    err.throw()
    print(f"regular positions shape: {regularly_sampled_rays.shape}")
    figure, axes = plt.subplots(1, 2, subplot_kw={"projection": "3d", "aspect": "equal"})
    figure.set_size_inches(14, 7)
    ax_rays_regular, ax_rays_nerf = axes
    ax_rays_regular.scatter([0], [0], [0], color="red")
    ax_rays_nerf.scatter([0], [0], [0], color="red")

    ax_rays_regular.set_title("rays towards pixels, regular sampling")
    ax_rays_regular.text(1, 0, 0, "x-axis", "x")  # type: ignore
    ax_rays_regular.text(0, 1, 0, "y-axis", "y")  # type: ignore
    ax_rays_regular.text(0, 0, 1, "z-axis", "z")  # type: ignore
    flat_positions = regularly_sampled_rays.reshape(-1, 4)
    flat_positions = flat_positions[:, :3] / flat_positions[:, 3:4]
    ax_rays_regular.scatter(
        flat_positions[:, 0],
        flat_positions[:, 1],
        flat_positions[:, 2],
    )
    plot_pixels(ax_rays_regular, all_pixel_xy)

    err, nerf_sampled_rays = checkify.checkify(
        sample_nerf_rendering_positions_along_rays
    )(
        rays_world_frame,
        rays_camera_frame.shape[0],
        0.5,
        5,
        7,
        jax.random.PRNGKey(0),
    )
    err.throw()
    print(f"nerf/random positions shape: {nerf_sampled_rays.shape}")
    ax_rays_nerf.set_title("rays towards pixels, NeRF (bins) sampling")
    ax_rays_nerf.text(1, 0, 0, "x-axis", "x")  # type: ignore
    ax_rays_nerf.text(0, 1, 0, "y-axis", "y")  # type: ignore
    ax_rays_nerf.text(0, 0, 1, "z-axis", "z")  # type: ignore
    flat_positions = nerf_sampled_rays.reshape(-1, 4)
    flat_positions = flat_positions[:, :3] / flat_positions[:, 3:4]
    ax_rays_nerf.scatter(
        flat_positions[:, 0],
        flat_positions[:, 1],
        flat_positions[:, 2],
    )
    plot_pixels(ax_rays_nerf, all_pixel_xy)
    plt.show()


test_sample_rays_from_actual_cameras()